<a href="https://colab.research.google.com/github/ssykes-eth/ETH_275-0005-00L/blob/code_exercises/01_react_sherlock_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🕵️ Sherlock Holmes with ReAct

Overnight at **Baskerville Tech**, a proprietary robotics design was copied to an unauthorized drive and sold to a competitor. Security narrows the theft to **12:30–2:00 AM**. Six people had motive, access, or both — but only hard evidence can separate rumor from guilt.

In this workshop, **you build the investigators**:

- First comes **Dr. Watson** — a standard LLM prompt that guesses from narrative profiles alone.
- Then enters **Sherlock Holmes** — a **ReAct** agent that thinks, calls corporate tools (logs, badges, emails, bank records), observes the results, and only then accuses.

**What you'll do next:** load the case, watch Watson jump to a confident (and likely wrong) answer, play the investigation yourself in an interactive game, then wire Sherlock's Thought → Action → Observation loop — first by hand, then driven by the LLM — and see why **Reason + Act** beats guessing when the stakes are real.

## 0.1 Setup

In [ ]:
# @title Setup: install the backend, fetch the case files, import dependencies { display-mode: "form" }
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path
from typing import Any, Dict, List

try:
    import transformers
except ImportError:
    print("installing transformers (the language model this notebook runs on) ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers"])

REPO_OWNER = "eth-fdd-fs26"
REPO_NAME  = "FDD-WE6-public"

def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

if _in_colab():
    url = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
    if os.path.isdir(REPO_NAME):
        print("Repo already present — refreshing to latest…")
        res = subprocess.run(["git", "-C", REPO_NAME, "pull", "-q", url], capture_output=True, text=True)
        if res.returncode != 0:
            print("  (could not pull — using the existing copy)")
    else:
        print("Cloning the exercise repo…")
        res = subprocess.run(["git", "clone", "-q", url], capture_output=True, text=True)
        if res.returncode != 0:
            tail = (res.stderr.strip().splitlines() or ["(no message)"])[-1]
            print("  clone failed:", tail)

# Move to the workshop folder (the folder holding corpus/) so case files and helpers resolve.
for _root in [
    os.path.join(REPO_NAME, "01_react_sherlock"),
    "01_react_sherlock",
    ".",
    os.path.dirname(os.getcwd()),
    os.getcwd(),
]:
    if os.path.exists(os.path.join(_root, "corpus", "case_data.json")):
        os.chdir(_root)
        break
else:
    raise FileNotFoundError(
        f"01_react_sherlock case data not found — the clone of {REPO_OWNER}/{REPO_NAME} did not land "
        "(see the message above). Check the Colab runtime has network access, then re-run this cell.")

ROOT = Path.cwd()
CORPUS = ROOT / "corpus"
ENV_DIR = str(ROOT)
# hf_llm.py lives in corpus/ — put that on the import path.
sys.path.insert(0, str(CORPUS))

DATA_PATH = CORPUS / "case_data.json"
with open(DATA_PATH, "r", encoding="utf-8") as f:
    CASE = json.load(f)

SUSPECTS = [s["name"] for s in CASE["suspects"]]

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:                             # torch missing, or a broken install
    DEVICE = "none"

print("Working directory:", os.getcwd(), "· helpers importable ✅")
print("case loaded from", ENV_DIR)
print("%d suspects, time window %s · case: %s"
      % (len(SUSPECTS), CASE["incident"]["time_window"], CASE["incident"]["name"]))
if DEVICE == "cuda":
    print("torch sees a GPU, so the language model later on will be comfortable.")
elif DEVICE == "cpu":
    print("torch sees no GPU. The model later on will run, slowly. Runtime, Change")
    print("runtime type, T4 takes a minute and is worth it.")
else:
    print("no torch here, so the model cannot load. There is no stand in: install torch.")

### Load the LLM model using API Key

Watson and Sherlock both call **Google Gemini** (e.g. `gemini-3.1-flash-lite`) with your API key — no local model download required.

**Add `GEMINI_API_KEY` in Google Colab:**

1. Open the **Secrets** panel in the left sidebar (🔑 icon).
2. Click **Add new secret**.
3. Name it exactly `GEMINI_API_KEY` and paste your Gemini API key as the value.
4. Enable **Notebook access** for that secret so this notebook can read it.
5. Run the API setup cell below — it loads the key via `google.colab.userdata.get("GEMINI_API_KEY")`.

Get a key from [Google AI Studio](https://aistudio.google.com/apikey). Free-tier models have a low calls-per-minute limit; the setup cell paces requests so you stay under quota.

In [ ]:
# @title API Setup and Model Loading {display-mode: "form"}
# Add GEMINI_API_KEY in Colab Secrets (key icon) — do not paste the key into this cell.
import os
import re
import subprocess
import sys
import time

# Install the Google GenAI SDK (provides: from google import genai)
try:
    from google import genai
except ImportError:
    print("Installing google-genai ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "google-genai"])
    from google import genai

MODEL = "gemini-3.1-flash-lite"
CALLS_PER_MINUTE = 10  # free-tier budget; raise on a paid key

USE_REAL_LLM = False
client = None

_key = None
try:
    from google.colab import userdata
    _key = userdata.get("GEMINI_API_KEY")
except Exception:
    _key = os.environ.get("GEMINI_API_KEY")

if _key:
    client = genai.Client(api_key=_key)
    USE_REAL_LLM = True
    print("connected to %s, free tier budget %d calls a minute" % (MODEL, CALLS_PER_MINUTE))
else:
    print("no GEMINI_API_KEY found")
    print("Add a Colab secret named GEMINI_API_KEY, enable Notebook access, then re-run.")

try:
    from google.genai import types
    _CFG = types.GenerateContentConfig(
        temperature=0.0,
        thinking_config=types.ThinkingConfig(thinking_budget=0),
    )
except Exception:
    _CFG = None

_call_times = []


def _wait_for_quota():
    """Stay under CALLS_PER_MINUTE in any rolling 60s window."""
    while True:
        now = time.time()
        while _call_times and now - _call_times[0] > 60:
            _call_times.pop(0)
        if len(_call_times) < CALLS_PER_MINUTE:
            _call_times.append(now)
            return
        wait = 60 - (now - _call_times[0]) + 1
        print("   (free tier: %d calls a minute. Waiting %.0f s.)" % (CALLS_PER_MINUTE, wait))
        time.sleep(wait)


def _retry_seconds(message, fallback):
    for pattern in (r"retryDelay['\"]?:\s*['\"]?(\d+(?:\.\d+)?)s", r"retry in (\d+(?:\.\d+)?)s"):
        m = re.search(pattern, message)
        if m:
            return float(m.group(1)) + 1
    return fallback


def call_gemini(prompt, retries=5):
    """One Gemini call with free-tier pacing and transient retries."""
    if not USE_REAL_LLM or client is None:
        raise RuntimeError("GEMINI_API_KEY not loaded — fix the secret, then re-run this cell.")
    delay, cfg = 10.0, _CFG
    for attempt in range(retries):
        _wait_for_quota()
        try:
            r = client.models.generate_content(model=MODEL, contents=prompt, config=cfg)
            return (r.text or "").strip()
        except Exception as e:
            s = str(e)
            if cfg is not None and ("thinking" in s.lower() or "INVALID_ARGUMENT" in s):
                cfg = None
                continue
            transient = any(
                t in s
                for t in ("429", "RESOURCE_EXHAUSTED", "503", "UNAVAILABLE", "500", "INTERNAL")
            )
            if not transient or attempt == retries - 1:
                raise
            wait = _retry_seconds(s, delay)
            print("   (the API asked us to slow down. Waiting %.0f s.)" % wait)
            time.sleep(wait)
            _call_times.clear()
            delay = min(delay * 2, 60)
    raise RuntimeError("gave up after %d attempts" % retries)


## 0.2 🥷 Problem at Hand

In [ ]:
# @title Case map of Baskerville Tech: Suspects, APIs, and Our Detectives { display-mode : 'form' }

from IPython.display import HTML, display

# Same SVG portraits as detective_game.html
_AVATARS = {
    "Alice": '<svg viewBox="0 0 120 120" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="Alice"><defs><linearGradient id="aBg" x1="0" y1="0" x2="1" y2="1"><stop offset="0%" stop-color="#2d4a6f"/><stop offset="100%" stop-color="#1a2f4a"/></linearGradient></defs><rect width="120" height="120" fill="url(#aBg)"/><circle cx="60" cy="48" r="28" fill="#f0c7a0"/><path d="M28 48c2-28 22-40 32-40s30 12 32 40c-8-10-20-14-32-14s-24 4-32 14z" fill="#2b1d14"/><rect x="38" y="46" width="44" height="10" rx="5" fill="#1e293b" opacity=".9"/><circle cx="48" cy="51" r="7" fill="none" stroke="#7eb8ff" stroke-width="2"/><circle cx="72" cy="51" r="7" fill="none" stroke="#7eb8ff" stroke-width="2"/><path d="M55 51h10" stroke="#7eb8ff" stroke-width="2"/><path d="M48 62c4 5 20 5 24 0" fill="none" stroke="#c9896a" stroke-width="2" stroke-linecap="round"/><path d="M20 120c8-28 28-40 40-40s32 12 40 40" fill="#334155"/><rect x="48" y="86" width="24" height="8" rx="2" fill="#7eb8ff"/></svg>',
    "Bob": '<svg viewBox="0 0 120 120" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="Bob"><defs><linearGradient id="bBg" x1="0" y1="0" x2="1" y2="1"><stop offset="0%" stop-color="#4a3d2d"/><stop offset="100%" stop-color="#2f2618"/></linearGradient></defs><rect width="120" height="120" fill="url(#bBg)"/><circle cx="60" cy="50" r="28" fill="#e8b892"/><path d="M30 52c0-26 14-38 30-38s30 12 30 38c-10-8-20-10-30-10s-20 2-30 10z" fill="#6b4423"/><circle cx="48" cy="50" r="3.5" fill="#2b1d14"/><circle cx="72" cy="50" r="3.5" fill="#2b1d14"/><path d="M50 66h20" fill="none" stroke="#b87454" stroke-width="2.5" stroke-linecap="round"/><path d="M18 120c10-30 28-42 42-42s32 12 42 42" fill="#c97848"/><circle cx="78" cy="88" r="10" fill="#c94c4c"/><text x="78" y="92" text-anchor="middle" font-size="8" fill="#fff" font-family="sans-serif" font-weight="700">QA</text></svg>',
    "Charlie": '<svg viewBox="0 0 120 120" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="Charlie"><defs><linearGradient id="cBg" x1="0" y1="0" x2="1" y2="1"><stop offset="0%" stop-color="#2d4a3d"/><stop offset="100%" stop-color="#1a3026"/></linearGradient></defs><rect width="120" height="120" fill="url(#cBg)"/><circle cx="60" cy="52" r="27" fill="#d4a574"/><path d="M33 50c2-22 16-32 27-32s25 10 27 32c-8-6-17-8-27-8s-19 2-27 8z" fill="#1f1a14"/><circle cx="48" cy="50" r="3" fill="#2b1d14"/><circle cx="72" cy="50" r="3" fill="#2b1d14"/><circle cx="48" cy="50" r="8" fill="none" stroke="#111827" stroke-width="3"/><circle cx="72" cy="50" r="8" fill="none" stroke="#111827" stroke-width="3"/><path d="M56 50h8" stroke="#111827" stroke-width="3"/><path d="M40 50h-6M86 50h-6" stroke="#111827" stroke-width="3" stroke-linecap="round"/><path d="M50 64c4 5 16 5 20 0" fill="none" stroke="#a86b45" stroke-width="2" stroke-linecap="round"/><path d="M20 120c8-26 26-38 40-38s32 12 40 38" fill="#166534"/><rect x="50" y="86" width="20" height="22" rx="2" fill="#fbbf24"/><circle cx="60" cy="97" r="5" fill="#166534"/></svg>',
    "Diana": '<svg viewBox="0 0 120 120" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="Diana"><defs><linearGradient id="dBg" x1="0" y1="0" x2="1" y2="1"><stop offset="0%" stop-color="#4a2d4a"/><stop offset="100%" stop-color="#2f1a2f"/></linearGradient></defs><rect width="120" height="120" fill="url(#dBg)"/><path d="M22 70c4-40 22-58 38-58s34 18 38 58c-12-16-26-22-38-22s-26 6-38 22z" fill="#1a0f14"/><circle cx="60" cy="50" r="27" fill="#e6b898"/><path d="M28 55c6-30 20-42 32-42s26 12 32 42" fill="#1a0f14"/><circle cx="48" cy="50" r="3.2" fill="#3b1d2e"/><circle cx="72" cy="50" r="3.2" fill="#3b1d2e"/><path d="M50 64c5 4 15 4 20 0" fill="none" stroke="#c9896a" stroke-width="2" stroke-linecap="round"/><path d="M18 120c10-28 28-40 42-40s32 12 42 40" fill="#7c3aed"/><path d="M48 88h24l-4 10h-16z" fill="#fbbf24"/></svg>',
    "Eve": '<svg viewBox="0 0 120 120" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="Eve"><defs><linearGradient id="eBg" x1="0" y1="0" x2="1" y2="1"><stop offset="0%" stop-color="#3d3d4a"/><stop offset="100%" stop-color="#262632"/></linearGradient></defs><rect width="120" height="120" fill="url(#eBg)"/><circle cx="60" cy="36" r="10" fill="#5b21b6"/><path d="M28 52c4-24 18-36 32-36s28 12 32 36c-8-8-18-12-32-12s-24 4-32 12z" fill="#5b21b6"/><circle cx="60" cy="52" r="26" fill="#d4a574"/><path d="M34 48c2-10 10-18 26-18s24 8 26 18" fill="#5b21b6"/><circle cx="48" cy="52" r="3" fill="#2b1d14"/><circle cx="72" cy="52" r="3" fill="#2b1d14"/><circle cx="48" cy="52" r="8" fill="none" stroke="#c4b5fd" stroke-width="2"/><circle cx="72" cy="52" r="8" fill="none" stroke="#c4b5fd" stroke-width="2"/><path d="M56 52h8" stroke="#c4b5fd" stroke-width="2"/><path d="M50 64c4 4 16 4 20 0" fill="none" stroke="#a86b45" stroke-width="2" stroke-linecap="round"/><path d="M20 120c8-26 26-38 40-38s32 12 40 38" fill="#4c1d95"/><path d="M36 88h48v8H36z" fill="#a78bfa" opacity=".85"/><rect x="52" y="96" width="16" height="12" rx="1" fill="#ddd6fe"/></svg>',
    "Frank": '<svg viewBox="0 0 120 120" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="Frank"><defs><linearGradient id="fBg" x1="0" y1="0" x2="1" y2="1"><stop offset="0%" stop-color="#4a4a2d"/><stop offset="100%" stop-color="#2f2f1a"/></linearGradient></defs><rect width="120" height="120" fill="url(#fBg)"/><circle cx="60" cy="52" r="27" fill="#dbb089"/><path d="M34 48c2-20 14-30 26-30s24 10 26 30c-8-4-16-6-26-6s-18 2-26 6z" fill="#9ca3af"/><path d="M38 42h44" stroke="#6b7280" stroke-width="4" stroke-linecap="round"/><circle cx="48" cy="52" r="3.2" fill="#2b1d14"/><circle cx="72" cy="52" r="3.2" fill="#2b1d14"/><path d="M52 66h16" stroke="#a86b45" stroke-width="2" stroke-linecap="round"/><path d="M20 120c8-26 26-38 40-38s32 12 40 38" fill="#374151"/><rect x="42" y="88" width="36" height="10" rx="2" fill="#eab308"/><text x="60" y="96" text-anchor="middle" font-size="7" fill="#1f2937" font-family="monospace" font-weight="700">IT</text></svg>',
}
_API_META = [
    ("🖥️", "query_server_logs", "Who touched which server, by 4-hour slot"),
    ("🪪", "check_badge_swipes", "Who is still badge-IN at a given HH:MM"),
    ("📧", "inspect_work_emails", "Flagged snippets from work mail"),
    ("🏦", "check_bank_records", "Wage vs deposits + flagged wires (motive)"),
]

_suspect_cards = []
for s in CASE["suspects"]:
    svg = _AVATARS.get(s["name"], "")
    _suspect_cards.append(
        f'<div class="cm-card">'
        f'<div class="cm-avatar">{svg}</div>'
        f'<div class="cm-name">{s["name"]}</div>'
        f'<div class="cm-role">{s["role"]}</div></div>'
    )

_api_rows = "".join(
    f'<div class="cm-api"><span class="cm-api-icon">{icon}</span>'
    f'<code>{name}</code><span class="cm-api-desc">{desc}</span></div>'
    for icon, name, desc in _API_META
)

_incident = CASE["incident"]
html = f"""
<style>
.cm-wrap {{
  font-family: "Source Sans 3", "Segoe UI", system-ui, sans-serif;
  color: #e8e6e3; background: #0f1419; border-radius: 14px;
  border: 1px solid rgba(212,168,83,.35); padding: 20px 22px 24px;
  background-image:
    radial-gradient(ellipse at 15% 0%, rgba(212,168,83,.10), transparent 45%),
    radial-gradient(ellipse at 90% 100%, rgba(91,141,239,.08), transparent 40%);
  max-width: 980px; margin: 8px 0 4px;
}}
.cm-wrap h3 {{
  font-family: Georgia, "Playfair Display", serif; color: #d4a853;
  margin: 0 0 6px; font-size: 1.25rem; letter-spacing: .02em;
}}
.cm-sub {{ color: #8b9cb3; font-size: .92rem; margin: 0 0 16px; line-height: 1.45; }}
.cm-label {{
  color: #d4a853; font-size: .72rem; letter-spacing: .14em;
  text-transform: uppercase; margin: 18px 0 10px; font-weight: 700;
}}
.cm-grid {{
  display: grid; grid-template-columns: repeat(auto-fill, minmax(140px, 1fr));
  gap: 10px;
}}
.cm-card {{
  background: #1a2332; border: 1px solid #2a3548; border-radius: 12px;
  padding: 12px 10px 14px; text-align: center;
  transition: border-color .15s ease, transform .15s ease;
  overflow: hidden;
}}
.cm-card:hover {{ border-color: #d4a853; transform: translateY(-2px); }}
.cm-avatar {{
  width: 88px; height: 88px; margin: 0 auto 8px; border-radius: 12px;
  overflow: hidden; border: 1px solid #2a3548;
}}
.cm-avatar svg {{ width: 100%; height: 100%; display: block; }}
.cm-name {{ font-weight: 700; color: #e8e6e3; }}
.cm-role {{ color: #8b9cb3; font-size: .78rem; margin-top: 2px; line-height: 1.3; }}
.cm-apis {{ display: flex; flex-direction: column; gap: 8px; }}
.cm-api {{
  display: flex; align-items: center; gap: 10px; flex-wrap: wrap;
  background: #1a2332; border: 1px solid #2a3548; border-radius: 10px;
  padding: 10px 12px;
}}
.cm-api-icon {{ font-size: 1.15rem; }}
.cm-api code {{
  color: #7eb8ff; background: #0f1419; padding: 2px 8px; border-radius: 6px;
  font-size: .82rem;
}}
.cm-api-desc {{ color: #8b9cb3; font-size: .85rem; }}
.cm-vs {{
  display: grid; grid-template-columns: 1fr 1fr; gap: 12px; margin-top: 4px;
}}
@media (max-width: 640px) {{ .cm-vs {{ grid-template-columns: 1fr; }} }}
.cm-panel {{
  border-radius: 12px; padding: 14px 16px; border: 1px solid #2a3548;
  background: #1a2332;
}}
.cm-panel.watson {{ border-color: rgba(201,76,76,.55); }}
.cm-panel.sherlock {{ border-color: rgba(76,175,122,.55); }}
.cm-panel h4 {{ margin: 0 0 8px; font-size: 1rem; }}
.cm-panel.watson h4 {{ color: #e07a7a; }}
.cm-panel.sherlock h4 {{ color: #6fcf97; }}
.cm-panel ul {{ margin: 0; padding-left: 18px; color: #c5cdd8; font-size: .88rem; line-height: 1.55; }}
.cm-tag {{
  display: inline-block; font-size: .68rem; letter-spacing: .08em;
  text-transform: uppercase; padding: 2px 8px; border-radius: 999px;
  margin-left: 6px; vertical-align: middle;
}}
.cm-tag.lock {{ background: rgba(201,76,76,.2); color: #e07a7a; }}
.cm-tag.open {{ background: rgba(76,175,122,.2); color: #6fcf97; }}
.cm-foot {{
  margin-top: 16px; padding-top: 12px; border-top: 1px solid #2a3548;
  color: #8b9cb3; font-size: .86rem; line-height: 1.45;
}}
.cm-foot strong {{ color: #d4a853; }}
</style>
<div class="cm-wrap">
  <h3>🕵️ Case Map — Baskerville Tech</h3>
  <p class="cm-sub">
    Breach window <strong style="color:#d4a853">{_incident["time_window"]}</strong>
    · {_incident["name"]} · six persons of interest, four corporate APIs
  </p>

  <div class="cm-label">The lineup</div>
  <div class="cm-grid">{"".join(_suspect_cards)}</div>

  <div class="cm-label">Magnifying glass — mock corporate APIs</div>
  <div class="cm-apis">{_api_rows}</div>

  <div class="cm-label">Who can touch what?</div>
  <div class="cm-vs">
    <div class="cm-panel watson">
      <h4>👨‍⚕️ Dr. Watson <span class="cm-tag lock">profiles only</span></h4>
      <ul>
        <li>Sees the six narrative profiles above</li>
        <li><strong>Cannot</strong> call any API</li>
        <li>Guesses from vibes, motives, and job titles</li>
        <li>Fast — and often confidently wrong</li>
      </ul>
    </div>
    <div class="cm-panel sherlock">
      <h4>🕵️ Sherlock Holmes <span class="cm-tag open">full toolkit</span></h4>
      <ul>
        <li>Same profiles <em>plus</em> all four APIs</li>
        <li>Thought → Action → Observation loop</li>
        <li>Cross-checks badges, logs, emails, banks</li>
        <li>Accuses only after evidence stacks up</li>
      </ul>
    </div>
  </div>

  <p class="cm-foot">
    <strong>Punchline:</strong> same incident, same suspects —
    Watson reads the gossip column; Sherlock pulls the badge reader, server logs, inbox, and ledger.
  </p>
</div>
"""
display(HTML(html))


You've met the suspects and seen the toolkit split: Watson gets gossip; Sherlock gets the badge reader, logs, inbox, and ledger.

Next we build both investigators and watch them work the same case:

1. **Dr. Watson (standard prompting)** — one-shot guess from narrative profiles alone  
2. **Sherlock Holmes (ReAct)** — Thought → Action → Observation with the corporate APIs

The parts you need to fill in with code will be indicated with `???` and marked with 🎯. Fill the missing blocks and execute the cells in a sequential order.

Start with Watson — the confident shortcut — so Sherlock's evidence trail lands with a contrast.

## 1. 🎯 Dr. Watson: Standard Prompting

In this act, we intentionally simulate a **non-tool-using** assistant.

The point is not whether it can sound convincing; the point is whether it can **verify** claims.

A model without tool access often overweights dramatic narrative clues (motives, conflicts, personality) and underweights verifiable operational evidence.

In [ ]:
# @title Quick narrative briefing of what is available to Watson { display-mode : 'form' }
print("=== CASE BRIEF ===")
print(CASE["incident"]["summary"])
print("\n=== SUSPECT PROFILES ===")
for s in CASE["suspects"]:
    print(f"- {s['name']} ({s['role']}): {s['profile']}")

In [ ]:
# 🎯 Standard prompting: accuse from profiles only (no tools)

import re

def build_standard_prompt(case: dict) -> str:
    """One-shot prompt: narrative profiles only — no logs, badges, emails, or banks."""
    lines = [
        f"- {s['name']} ({s['role']}): {s['profile']}"
        for s in case["suspects"]
    ]
    return (
        "You are a corporate investigator.\n"
        "You have ONLY the narrative profiles below. You cannot query logs, badges, "
        "emails, or bank records. Do not invent verified forensic evidence.\n\n"
        f"Incident: {case['incident']['summary']}\n"
        f"Time window: {case['incident']['time_window']}\n\n"
        "Suspects:\n"
        + "\n".join(lines)
        + "\n\nWho stole the algorithm? Reply in this format:\n"
        "Culprit: <name>\n"
        "Confidence: <low/medium/high>\n"
        "Reasoning: <2-3 sentences>"
    )


def parse_accusation(text: str, suspects: list) -> dict:
    """Pull Culprit / Confidence / Reasoning out of the model reply."""
    culprit_match = re.search(r"Culprit:\s*(.+)", text, re.I)
    confidence_match = re.search(r"Confidence:\s*(.+)", text, re.I)
    reasoning_match = re.search(r"Reasoning:\s*(.+)", text, re.I | re.S)

    culprit = (culprit_match.group(1).strip() if culprit_match else "")
    for name in suspects:
        if name.lower() in culprit.lower():
            culprit = name
            break
    if culprit not in suspects:
        culprit = suspects[0]

    return {
        "culprit": culprit,
        "confidence": (confidence_match.group(1).strip() if confidence_match else "unknown"),
        "explanation": (reasoning_match.group(1).strip() if reasoning_match else text.strip()),
        "raw_response": text,
    }


def run_standard_prompt(case: dict) -> dict:
    """Call Gemini with the profile-only prompt and parse the accusation."""
    if not USE_REAL_LLM or client is None:
        raise RuntimeError(
            "GEMINI_API_KEY not loaded. Re-run the API setup cell after adding the Colab secret."
        )
    # 🎯🎯 pass the case data to the prompt using the build_standard_prompt function
    prompt = ???
    # 🎯🎯 call the gemini model with the prompt using the call_gemini function
    response = ???
    suspects = [s["name"] for s in case["suspects"]]
    # 🎯🎯 parse the response using the parse_accusation function
    return ???


standard_result = run_standard_prompt(CASE)

print("Accusation:", standard_result["culprit"])
print("Confidence:", standard_result["confidence"])
print("Reasoning:", standard_result["explanation"])


In [ ]:
# @title Check standard-prompting accusation vs ground truth { display-mode : 'form' }

truth = CASE["ground_truth"]["culprit"]
guess = standard_result["culprit"]
match = guess.strip().lower() == truth.strip().lower()

print("Standard prompting accusation:", guess)
print("Is this the real culprit?", match)
if not match:
    print(
        "\nProfiles-only prompting missed the mark — "
        "narrative vibes are not the same as verified evidence."
    )
else:
    print(
        "\nLucky hit on the name, but the reasoning still was not "
        "backed by logs, badges, emails, or bank records."
    )


### Why Standard Prompting Is Insufficient

Watson can only infer from the suspect bios. It does **not** check:

- Which credentials accessed sensitive systems in the breach window,
- Who was physically present,
- Who communicated suspicious intent,
- Who received suspicious payments.

That missing verification step is exactly what ReAct fixes.

## 2 Sherlock Holmes: ReAct

In [ ]:
# @title ReAct loop — Thought → Action → Observation { display-mode : 'form' }

from IPython.display import HTML, display

display(HTML("""
<style>
.react-wrap {
  font-family: "Source Sans 3", "Segoe UI", system-ui, sans-serif;
  color: #e8e6e3; background: #0f1419; border-radius: 14px;
  border: 1px solid rgba(212,168,83,.35); padding: 20px 22px 22px;
  background-image:
    radial-gradient(ellipse at 12% 0%, rgba(212,168,83,.10), transparent 45%),
    radial-gradient(ellipse at 88% 100%, rgba(91,141,239,.08), transparent 40%);
  max-width: 980px; margin: 8px 0 4px;
}
.react-wrap h3 {
  font-family: Georgia, "Playfair Display", serif; color: #d4a853;
  margin: 0 0 6px; font-size: 1.2rem;
}
.react-sub { color: #8b9cb3; font-size: .9rem; margin: 0 0 18px; line-height: 1.45; }
.react-flow {
  display: flex; flex-wrap: wrap; align-items: stretch; justify-content: center;
  gap: 10px;
}
.react-step {
  flex: 1 1 180px; max-width: 240px;
  background: #1a2332; border: 1px solid #2a3548; border-radius: 12px;
  padding: 14px 14px 16px; position: relative;
  animation: react-pop .55s ease both;
}
.react-step:nth-child(1) { animation-delay: .05s; border-color: rgba(126,184,255,.45); }
.react-step:nth-child(3) { animation-delay: .18s; border-color: rgba(212,168,83,.5); }
.react-step:nth-child(5) { animation-delay: .31s; border-color: rgba(111,207,151,.45); }
@keyframes react-pop {
  from { opacity: 0; transform: translateY(8px); }
  to { opacity: 1; transform: none; }
}
.react-badge {
  display: inline-block; font-size: .68rem; letter-spacing: .12em;
  text-transform: uppercase; font-weight: 700; padding: 3px 8px;
  border-radius: 999px; margin-bottom: 8px;
}
.react-step:nth-child(1) .react-badge { background: rgba(126,184,255,.18); color: #7eb8ff; }
.react-step:nth-child(3) .react-badge { background: rgba(212,168,83,.18); color: #d4a853; }
.react-step:nth-child(5) .react-badge { background: rgba(111,207,151,.18); color: #6fcf97; }
.react-icon { font-size: 1.6rem; margin-bottom: 4px; }
.react-title { font-weight: 700; font-size: 1.05rem; margin-bottom: 6px; }
.react-body { color: #8b9cb3; font-size: .86rem; line-height: 1.45; }
.react-eg { margin-top: 10px; background: #0f1419; border-radius: 8px;
  padding: 8px 10px; font-family: ui-monospace, SFMono-Regular, Menlo, monospace;
  font-size: .75rem; color: #c5cdd8; line-height: 1.4; border: 1px solid #2a3548;
}
.react-arrow {
  display: flex; align-items: center; justify-content: center;
  color: #d4a853; font-size: 1.4rem; flex: 0 0 auto; min-width: 28px;
  animation: react-pop .55s ease both;
}
.react-arrow:nth-child(2) { animation-delay: .12s; }
.react-arrow:nth-child(4) { animation-delay: .25s; }
.react-loop {
  margin-top: 16px; text-align: center; color: #8b9cb3; font-size: .88rem;
  padding-top: 12px; border-top: 1px solid #2a3548; line-height: 1.5;
}
.react-loop strong { color: #d4a853; }
.react-cycle {
  display: inline-block; margin-left: 6px;
  animation: react-spin 3.5s linear infinite;
}
@keyframes react-spin { to { transform: rotate(360deg); } }
@media (max-width: 700px) {
  .react-arrow { transform: rotate(90deg); width: 100%; min-height: 24px; }
}
</style>
<div class="react-wrap">
  <h3>🔁 The ReAct cycle</h3>
  <p class="react-sub">
    Sherlock does not guess once and stop — he loops until the evidence earns a Final Answer.
  </p>
  <div class="react-flow">
    <div class="react-step">
      <div class="react-badge">Step 1</div>
      <div class="react-icon">💭</div>
      <div class="react-title">Thought</div>
      <div class="react-body">What do I still need to learn? Narrow the question before touching a tool.</div>
      <div class="react-eg">“Who was on-site at 01:00 during the breach?”</div>
    </div>
    <div class="react-arrow">→</div>
    <div class="react-step">
      <div class="react-badge">Step 2</div>
      <div class="react-icon">🛠️</div>
      <div class="react-title">Action</div>
      <div class="react-body">Pick one corporate API and a precise input. One tool call per turn.</div>
      <div class="react-eg">check_badge_swipes("01:00")</div>
    </div>
    <div class="react-arrow">→</div>
    <div class="react-step">
      <div class="react-badge">Step 3</div>
      <div class="react-icon">👁️</div>
      <div class="react-title">Observation</div>
      <div class="react-body">Read what the tool returned — facts only, no inventing evidence.</div>
      <div class="react-eg">["Alice", "Charlie", "Frank"]</div>
    </div>
  </div>
  <p class="react-loop">
    Then <strong>loop</strong>: Observation feeds the next Thought → Action → Observation
    <span class="react-cycle">↻</span><br/>
    until opportunity + action + motive support a justified Final Answer.
  </p>
</div>
"""))


In [ ]:
# @title Build the Tool layer (mock corporate APIs) { display-mode : 'form' }

def query_server_logs(time_window: str) -> Any:
    """Query server access logs for a 4-hour slot.

    Available slots: 00:00-04:00, 04:00-08:00, 08:00-12:00,
    12:00-16:00, 16:00-20:00, 20:00-00:00.

    The reported theft (00:30-02:00) falls in slot 00:00-04:00.
    """
    logs = CASE["server_logs"].get(time_window)
    if logs is None:
        return {
            "error": f"No logs for '{time_window}'.",
            "available_slots": list(CASE["server_logs"].keys()),
            "hint": "Reported theft is 00:30-02:00 → query '00:00-04:00'.",
        }
    return logs


def _hhmm_to_minutes(time_str: str) -> int:
    """Parse HH:MM into minutes since midnight."""
    parts = time_str.strip().split(":")
    if len(parts) != 2:
        raise ValueError(f"expected HH:MM, got {time_str!r}")
    hour, minute = int(parts[0]), int(parts[1])
    if not (0 <= hour <= 23 and 0 <= minute <= 59):
        raise ValueError(f"out-of-range time: {time_str!r}")
    return hour * 60 + minute


def check_badge_swipes(time: str) -> Any:
    """Return who is still badge-IN (on premises) at HH:MM.

    Overnight shifts are supported when IN is later than OUT
    (e.g. Charlie: 22:58 IN, 07:12 OUT). Someone who OUT at exactly
    `time` is treated as already gone.

    Example: check_badge_swipes("01:00") during the theft window.
    """
    try:
        query = _hhmm_to_minutes(time)
    except ValueError:
        return {
            "error": f"Invalid time {time!r}. Use HH:MM, e.g. '01:00'.",
            "hint": "Try a time inside the breach window 00:30-02:00.",
        }

    active: List[str] = []
    for name, events in CASE["badge_swipes"].items():
        in_t = out_t = None
        for event in events:
            bits = event.split()
            if len(bits) != 2:
                continue
            stamp, kind = bits[0], bits[1].upper()
            if kind == "IN":
                in_t = _hhmm_to_minutes(stamp)
            elif kind == "OUT":
                out_t = _hhmm_to_minutes(stamp)
        if in_t is None or out_t is None:
            continue
        # Same-calendar-day shift vs overnight (IN after OUT on the clock).
        if in_t <= out_t:
            on_site = in_t <= query < out_t
        else:
            on_site = query >= in_t or query < out_t
        if on_site:
            active.append(name.title())
    return active


def inspect_work_emails(employee_name: str) -> List[str]:
    return CASE["emails"].get(employee_name, [])


def check_bank_records(employee_name: str) -> Dict[str, Any]:
    return CASE["bank_records"].get(employee_name, {})


TOOLS = {
    "query_server_logs": query_server_logs,
    "check_badge_swipes": check_badge_swipes,
    "inspect_work_emails": inspect_work_emails,
    "check_bank_records": check_bank_records,
}

print("Tools available:")
for name in TOOLS:
    print("-", name)

## 2.1 Interactive Detective Game

Before we hand the case to an LLM, **you** play Sherlock.

Use the embedded game below to:

1. Review the **six suspect profiles**
2. Query the **four corporate APIs** and read each Observation
3. Build an investigation log as you go (Thought → Action → Observation)
4. Submit a **final accusation** when the evidence stacks up

Your investigation log is the human version of what ReAct asks a model to produce: a trail of reasoning and tool calls that earns the verdict, instead of a one-shot guess from vibes.


In [ ]:
# @title Game Window { display-mode : 'form' }

import html as html_lib
from IPython.display import HTML, display

GAME_PATH = CORPUS / "detective_game.html"
if not GAME_PATH.exists():
    raise FileNotFoundError(f"Game file not found: {GAME_PATH}")

EMBED_HEIGHT = 700
game_html = GAME_PATH.read_text(encoding="utf-8")
iframe = (
    '<iframe srcdoc="' + html_lib.escape(game_html, quote=True) + '" '
    'width="100%" height="' + str(EMBED_HEIGHT) + '" '
    'style="width:100%;height:' + str(EMBED_HEIGHT) + 'px;'
    'border:1px solid #2a3548;border-radius:10px;background:#0f1419;'
    'display:block;overflow:auto">'
    '</iframe>'
)
display(HTML(iframe))


In [ ]:
# @title Run if game window is not rendering { display-mode : 'form' }

print("If the window below is empty, open folder icon on the sidebar and download the file:")
print(" ", GAME_PATH.resolve())
print("Then open corpus/detective_game.html for the full-page version.")


Were you able to find the culprit? If yes, amazing! Seems like we have a Sherlock among ourselves!

If no, don't worry, let's see if our ReAct Sherlock can help us out.  

## 2.2 Build ReAct from Scratch

In the game window, you drove **Thought → Action → Observation** by hand.
Now the **LLM** chooses each Action; you wire the loop yourself.

You need three pieces:

1. **Ask the model** for the next Thought + Action + Action Input (or Final Answer)
2. **Execute** that Action with our tools and capture the Observation
3. **Loop**: append Observation to a scratchpad, ask again, stop when the model gives a Final Answer


### 🎯 2.2.1 Ask the LLM what to do next

The model must reply in ReAct format — either a tool call, or a final accusation:

```text
Thought: ...
Action: <tool_name>
Action Input: <plain string>
```

or

```text
Thought: ...
Final Answer: <suspect name>
```

First we define the different prompt design components, the instructions we want our LLM to follow.

In [ ]:
# @title Instruction Components for the ReAct prompt
from hf_llm import format_suspects, normalize_culprit, parse_react_step

MAX_REACT_STEPS = 10

INVESTIGATION_POLICY = """
Hard rules:
- Do not Final Answer until opportunity AND motive are supported by Observations in the log.
"""

TOOL_DESCRIPTIONS = """
- query_server_logs(time_window): server access in a 4-hour slot ONLY
  (valid: 00:00-04:00, 04:00-08:00, 08:00-12:00, 12:00-16:00, 16:00-20:00, 20:00-00:00)
- check_badge_swipes(time): who is still badge-IN at HH:MM in 24-hour format (e.g. 01:00)
- inspect_work_emails(employee_name): recent email snippets for one employee
- check_bank_records(employee_name): wage vs deposit/withdraw + flagged wire
"""

REACT_FORMAT = """
Respond with exactly ONE of these formats per turn — never both.
Never invent Observations.

Tool call:
Thought: <your reasoning>
Action: <exact lowercase tool name>
Action Input: <plain string, e.g. 01:00 or 04:00-08:00 or Alice>

Verdict (only after opportunity + motive are in the log):
Thought: <confirm opportunity + suspicious action + motive from Observations>
Final Answer: <one suspect name only, e.g. Charlie>

Rules:
- Thought, then either Action + Action Input OR Final Answer — never both.
- After Action Input, stop.
- Action Input must be a plain string, not JSON.
"""

Now let's bring the different instruction components together to build a holistic prompt.

In [ ]:
def build_system_prompt() -> str:
    return (
        "You are Sherlock Holmes, a ReAct investigator at Baskerville Tech.\n"
        "Gather evidence with tools before accusing anyone.\n"
        "Each reply is ONE step only.\n"
        f"Available tools:\n{???}\n" # 🎯🎯 fill in the tool descriptions from the instruction components
        f"{???}\n" # 🎯🎯 fill in the investigation policy from the instruction components
        f"{???}" # 🎯🎯 fill in the react format from the instruction components
    )


def build_user_prompt(scratchpad: str) -> str:
    # 🎯🎯 build the user prompt using the scratchpad
    log = ??? or (
        '(none yet — start with check_badge_swipes("01:00"),'
         'then query server logs, query emails, and check bank records)'
    ) # given: the fallback message for the first turn
    return (
        f"Incident: {CASE['incident']['summary']}\n" # given: the incident summary
        f"Breach window: {CASE['incident']['time_window']}\n\n" # given: the breach window
        f"Suspects:\n{format_suspects(CASE)}\n\n" # given: the suspects
        f"Investigation log so far:\n{log}\n\n" # 🎯🎯 fill in the investigation log so far
        "Reply with your next single step only "
        "(Thought + Action + Action Input, or Thought + Final Answer).\n"
        "If opportunity or motive is still missing from the log, call a tool next — "
        "do not Final Answer yet."
    )


def llm_decide(scratchpad: str) -> dict:
    """Ask Gemini for the next ReAct step and parse Action / Action Input / Final Answer."""
    # 🎯🎯 build the prompt using the build_system_prompt and build_user_prompt functions
    prompt = (
        ???
    )
    raw = call_gemini(prompt)
    step = parse_react_step(raw)
    step["raw"] = raw
    return step

In [ ]:
# @title Sanity check: one undecided step (no tools run yet)
_demo = llm_decide("")
print("Demo Thought:", _demo.get("thought"))
print("Demo Action:", _demo.get("action"))
print("Demo Action Input:", _demo.get("action_input"))
print("Demo Final Answer:", _demo.get("final_answer"))

### 🎯 2.2.2 Execute the chosen action

Whatever the model asked for, look it up in `TOOLS`, run it, and return the Observation.
Unknown tools become an error Observation (fed back to the model on the next turn).


In [ ]:
# Run one tool call and return an Observation
def execute_action(action: str | None, action_input: Any) -> Any:
    """Execute the LLM's chosen tool call. Always returns something the model can read."""
    if not action or action not in TOOLS:
        return {
            "error": f"Invalid or missing action: {action!r}.",
            "available_tools": list(TOOLS),
        }
    try:
        # 🎯🎯 execute the action using the tools dictionary
        return ???
    except Exception as exc:  # noqa: BLE001 — surface tool errors as observations
        return {"error": str(exc)}

In [ ]:
# @title Sanity check against a known slot
print(execute_action("query_server_logs", "00:00-04:00"))
print(execute_action("not_a_real_tool", "Alice"))

### 🎯 2.2.3 — The ReAct loop

Until the model issues a **Final Answer** (or you hit the step budget):

1. `llm_decide(scratchpad)` → Thought / Action / Action Input
2. `execute_action(...)` → Observation
3. Append Thought, Action, Action Input, Observation to the scratchpad
4. Repeat — the Observation is now part of the next prompt


In [ ]:
# Full ReAct loop: decide → act → observe → repeat

def run_react_from_scratch(max_steps: int = MAX_REACT_STEPS) -> dict:
    suspects = [s["name"] for s in CASE["suspects"]]
    scratchpad = ""
    trace = []
    culprit = None

    for step_i in range(1, max_steps + 1):
        decision = ??? # 🎯🎯 ask the LLM for the next step using the llm_decide function
        thought = ??? or "Continuing investigation." # 🎯🎯 get the thought from the decision
        action = ??? # 🎯🎯 get the action from the decision
        action_input = ??? # 🎯🎯 get the action input from the decision
        if action_input is not None:
            action_input = action_input.lower()
        final_answer = ??? # 🎯🎯 get the final answer from the decision

        print(f"\n=== Step {step_i} ===")
        print("Thought:", thought)

        if final_answer: #given: if the final answer is not None
            culprit = normalize_culprit(final_answer, suspects) or final_answer
            print("Final Answer:", culprit)
            decision["observation"] = {"final_answer": final_answer}
            trace.append(decision)
            scratchpad += f"\nThought: {thought}\nFinal Answer: {final_answer}\n"
            break

        print("Action:", action)
        print("Action Input:", action_input)

        observation = ??? # 🎯🎯 execute the action using the execute_action function
        print("Observation:", observation)

        scratchpad += (
            f"\nThought: {thought}\n"
            f"Action: {action}\n"
            f"Action Input: {action_input}\n"
            f"Observation: {observation}\n"
        )
        decision["observation"] = ??? # 🎯🎯 add the observation to the decision
        trace.append(decision)

    if culprit is None: #given: if the culprit is None
        print("\n=== Forced final accusation ===")
        forced_prompt = (
            build_user_prompt(scratchpad)
            + "\n\nYou must now accuse exactly one suspect.\n"
            "Thought must confirm opportunity + suspicious action + motive "
            "from the investigation log (do not exonerate then accuse).\n"
            "Reply with Thought + Final Answer only."
        )
        raw = call_gemini(
            build_system_prompt()
            + "\n\n---\n\n"
            + forced_prompt
        )
        decision = parse_react_step(raw)
        decision["raw"] = raw
        final_answer = decision.get("final_answer") or raw
        culprit = normalize_culprit(final_answer, suspects) or suspects[0]
        print("Thought:", decision.get("thought"))
        print("Final Answer:", culprit)
        decision["observation"] = {"final_answer": final_answer}
        trace.append(decision)

    return {"culprit": culprit, "trace": trace, "scratchpad": scratchpad}


react_result  = run_react_from_scratch()
print("\nSherlock's culprit:", react_result["culprit"])
print("Ground Truth:", CASE["ground_truth"]["culprit"])
print("Match:", react_result["culprit"] == CASE["ground_truth"]["culprit"])


Was ReAct Sherlock able to catch the real culprit? The Stepwise rollout gives a clear view of the model's reasoning and action process to reach the final answer.

If you have time, you can play around with the Instruction components, the system prompt, and the user prompt to see how prompting can influence the ReAct results.

## Recap

### What changed between Watson and Sherlock?

- **Watson** optimized for a plausible story from profiles alone.
- **Sherlock** optimized for verifiable evidence via tool calls.

### Core concept takeaway

ReAct is not just "better prompting." It changes the operating model from one-shot guessing to iterative investigation with tool-grounded observations at every step.